# 07.1. Pipeline: inference strategies

Test-time variants of the best model of each family, without retraining:
- **Flip:** the mirrored frame as a second view.
- **Tiles:** four overlapping crops at 60% of the frame, so small objects are seen larger.
- **Merging** of the views by NMS or Weighted Boxes Fusion.
- **Low resolution:** every variant also on frames shrunk to 960 px, like the low-resolution test clips.

All on held-out fold 0 (every 5th frame), with the fold-0 models. A variant is kept for a model only if it beats
the single full-frame pass at both resolutions and in most clips. The change in mAP is the effect of inference
alone; the model stays the same.

## Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/core").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working so it is not saved as output
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q ultralytics faster-coco-eval

In [ ]:
import json

import pandas as pd
import torch

from core.cv import best_per_family, environment
from core.data import assign_folds, every_nth, find_data_root, load_pool
from core.variants import choose_variant, collect_view, compare_variants, make_detector, variant_dets

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
environment()

## Data and folds

In [ ]:
DATA = find_data_root()
frames, gt = load_pool(DATA)
frames["fold"] = frames["video"].map(assign_folds(frames))
held0 = every_nth(frames[frames["fold"] == 0], 5)  # fold-0 models never saw these frames
print(len(held0), "held-out frames of fold 0,", held0["video"].nunique(), "clips")

## Models and variants
The best run of each family on fold 0 (outputs of 02–06 and 07.0 attached as input).

In [ ]:
CANDIDATES = list(best_per_family(["baseline", "yolo_", "rtdetr", "frcnn", "retina", "fcos"], WORK).values())
print("models:", CANDIDATES)

LOW_RES = 960  # test has 680 and 960 px clips, the labelled data starts at 1344
REF = "full"
VARIANTS = {
    REF: dict(views=["full"]),
    "flip_nms65": dict(views=["full", "flip"], iou=0.65),
    "tiles_nms65": dict(views=["full", "tiles"], iou=0.65),
    "flip_tiles_nms65": dict(views=["full", "flip", "tiles"], iou=0.65),
    "flip_tiles_wbf65": dict(views=["full", "flip", "tiles"], iou=0.65, fuse="wbf"),
}
VIEWS = sorted({v for var in VARIANTS.values() for v in var["views"]})

## Variants on fold 0: native resolution and shrunk to 960 px

In [ ]:
paths = held0["path"].tolist()
results = {}
for name in CANDIDATES:
    detect = make_detector(name, 0, WORK)
    native_views = {v: collect_view(detect, paths, v) for v in VIEWS}
    low_views = {v: collect_view(detect, paths, v, max_side=LOW_RES) for v in VIEWS}
    native = compare_variants(native_views, VARIANTS, held0, gt, REF)
    lowres = compare_variants(low_views, VARIANTS, held0, gt, REF, clips=False)
    chosen = choose_variant(native, lowres, REF)
    variant_dets(native_views, VARIANTS[chosen]).to_parquet(WORK / f"fold0_dets_{name}.parquet")
    variant_dets(low_views, VARIANTS[chosen]).to_parquet(WORK / f"fold0_dets_{name}_lowres.parquet")
    results[name] = dict(native=native, lowres=lowres, chosen=chosen)
    print(name, "| chosen:", chosen)
    display(native.join(lowres[["mAP"]].rename(columns={"mAP": "mAP_960px"})))
    del detect
    torch.cuda.empty_cache()

## Summary: effect of inference per model
`gain` is the chosen variant minus the single full-frame pass, at native resolution and at 960 px.

In [ ]:
summary = pd.DataFrame({
    n: {"chosen": r["chosen"],
        "mAP_full": r["native"].loc[REF, "mAP"], "mAP": r["native"].loc[r["chosen"], "mAP"],
        "mAP_960_full": r["lowres"].loc[REF, "mAP"], "mAP_960": r["lowres"].loc[r["chosen"], "mAP"],
        "ms_per_frame": r["native"].loc[r["chosen"], "ms_per_frame"]}
    for n, r in results.items()}).T
num = summary.columns.drop("chosen")
summary[num] = summary[num].astype(float)
summary["gain"] = summary["mAP"] - summary["mAP_full"]
summary["gain_960"] = summary["mAP_960"] - summary["mAP_960_full"]
summary = summary.sort_values("mAP", ascending=False)
summary.to_csv(WORK / "inference_summary.csv")
(WORK / "chosen_variants.json").write_text(json.dumps(
    {n: VARIANTS[r["chosen"]] for n, r in results.items()}, indent=1))
summary

### Findings
*(After the run: which variants help which models, at what cost in time, and whether the gain holds at 960 px.)*